## Preliminaries
### Import statements

In [1]:
from osfclient import OSF
import pandas as pd
import os

### Global values

In [2]:
OSF_PROJECT = "uz6hg"
TEMP_DIR = "temp"
OSF_FILES = ["tokens.tsv", "accent_corrections.tsv", "lemma_corrections.tsv"]

### Check for temporary directory; create if necessary

To avoid overwriting output from other notebooks, we're saving to a temporary directory.

In [3]:
if not os.path.exists(TEMP_DIR):
    os.mkdir(TEMP_DIR)

## Load token data
### Download data files from OSF

In [4]:
osf = OSF()

project = osf.project(OSF_PROJECT)
storage = project.storage()
for file in storage.files:
    if file.name in OSF_FILES:
        with open(os.path.join(TEMP_DIR, file.name), "wb") as fh:
            file.write_to(fh)

100%|██████████████████████████████████| 73.1M/73.1M [00:04<00:00, 15.9Mbytes/s]
100%|██████████████████████████████████| 9.41k/9.41k [00:00<00:00, 10.8Mbytes/s]
100%|██████████████████████████████████| 39.9k/39.9k [00:00<00:00, 16.6Mbytes/s]


### Load the token table

In [12]:
tokens = pd.read_csv(os.path.join(TEMP_DIR, "tokens.tsv"), sep="\t", dtype=str)

### Corrections

In [13]:
# remove punctuation
tokens = tokens.loc[~(tokens["pos"]=="PUNCT")]

# correct elisions and accents
for corr_file in ["accent_corrections.tsv", "lemma_corrections.tsv"]:
    with open(os.path.join(TEMP_DIR, corr_file)) as f:
        for row in f.readlines():
            if not row.startswith("#"):
                old, new = row.strip().split("\t")
                tokens.loc[tokens["lemma"]==old, "lemma"] = new

# force Triphiodorus pref to string
tokens.loc[tokens["work"]=="Sack of Troy", "pref"] = " "

### Lemma count

In [14]:
corpus_lemma_count = tokens["lemma"].value_counts()

In [15]:
corpus_lemma_count

lemma
καί             13671
δʼ              11523
δέ               9816
ὁ                9589
τε               4038
                ...  
ἕρκεʼ               1
προμολαί            1
ἔμμορα              1
κατειλύω            1
ἀρειοτέροισι        1
Name: count, Length: 36652, dtype: int64

## Lemma analysis
### Hapax legomena per author

In [27]:
authors = ["Homer", "Apollonius", "Triphiodorus", "Quintus", "Nonnus"]
hapax_legomena = corpus_lemma_count[corpus_lemma_count==1].index.values
df = pd.DataFrame(dict(
    hapax = tokens.loc[tokens.lemma.isin(hapax_legomena), "author"].value_counts()[authors],
    tokens = tokens["author"].value_counts()[authors],
    ))
df["freq"] = df["hapax"].div(df["tokens"]) * 1000
display(df)

,hapax,tokens,freq
author,,,
Homer,6729,199029,33.809143
Apollonius,3092,38820,79.649665
Triphiodorus,373,4232,88.137996
Quintus,2850,60116,47.408344
Nonnus,7043,127577,55.205876


### Author-specific lemmata 
#### Including hapax legomena

In [108]:
lemma_count_by_author = pd.crosstab(tokens["lemma"], tokens["work"]).loc[corpus_lemma_count.index.values]

single_auth_lemmas = (lemma_count_by_author
    .gt(0) # true if author uses lemma, false otherwise
    .sum(axis=1) # count how many authors use each lemma
    .loc[lambda s: s==1] # filter for lemmas with single author
    .index.values # keep just the list of lemmas
)

df = (lemma_count_by_author
        .loc[single_auth_lemmas] # select only single-author lemmas
        .assign(_key=lambda d: d.sum(axis=1)) # temp column for sorting by count in any author
        .sort_values("_key", ascending=False) # sort
        .drop(columns="_key") # get rid of temp column after sorting
)

df.to_clipboard()

In [107]:
bool_df = lemma_count_by_author.gt(0).astype(int)
(bool_df.T @ bool_df).div(bool_df.sum())

work,Argonautica,Dionysiaca,Iliad,Odyssey,Posthomerica,Sack of Troy
work,,,,,,
Argonautica,1.000000,0.181935,0.303865,0.294849,0.325317,0.496777
Dionysiaca,0.366330,1.000000,0.334926,0.300661,0.355408,0.645017
Iliad,0.343875,0.188241,1.000000,0.388555,0.384196,0.563708
Odyssey,0.345874,0.175162,0.402763,1.000000,0.348300,0.527516
Posthomerica,0.322831,0.175162,0.336900,0.294648,1.000000,0.531978
Sack of Troy,0.117799,0.075962,0.118118,0.106635,0.127118,1.000000
